<div style="
    background-color:#2c3e50; 
    color:#ecf0f1; 
    font-weight:bold; 
    padding:20px 30px; 
    font-size:20px; 
    border-radius:8px; 
    text-align:center;
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    letter-spacing: 0.5px;
">
    Calculating extra statistical properties and consumption of water and energy
</div>

<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    This generates extra properties of each cycle and appends them to aggregated_data file creating a summary about the experiment
</h3>

In [ ]:
import numpy as np
from datetime import datetime
import pandas as pd

In [ ]:
# Load aggregated data
aggregated = pd.read_csv("aggregated_data.csv")

In [ ]:
# --- Config -----------------------------------------------------------------

STAT_FUNCS = {
    "mean":  np.nanmean,
    "median": np.nanmedian,
    "variance": np.nanvar,
    "std":   np.nanstd,
    "max":   np.nanmax,
    "min":   np.nanmin,
    "range": lambda x: np.nanmax(x) - np.nanmin(x),
}

FEATURE_SETS = {
    "wm": [
        "power", "current", "frequency", "power_factor", "voltage",
        "drum_door_temp", "drum_temperature",
        "water_temperature_in", "water_temperature_out",
        "ambient_temperature_1", "ambient_temperature_2",
        "ambient_humidity_1", "ambient_humidity_2",
        "pressure",
        # "w_oulet"  # intentionally excluded / dropped upstream
    ],
    "dm": [
        "current", "frequency", "power", "power_factor", "voltage",
        "inside_temperature", "inside_temperature_2", "inside_humidity",
        "ambient_temperature_1", "ambient_temperature_2",
        "ambient_humidity_1", "ambient_humidity_2",
    ],
}

VERBOSE = True  # print progress
UPDATE = False  # if True, will overwrite existing values in aggregated


# --- Helpers ----------------------------------------------------------------

def count_time(timestamps):
    """Return difference in minutes between first and last timestamp."""
    first = timestamps[0]
    last = timestamps[-1]

    # Convert to datetime if they are numeric UNIX timestamps
    if isinstance(first, (int, float)):
        first = datetime.fromtimestamp(first)
        last = datetime.fromtimestamp(last)

    difference_in_minutes = abs((last - first).total_seconds() / 60)
    return difference_in_minutes

def calculate_water(water_array):
    """Calculate total water consumption in liters from an array of water flow rates in ml."""
    water_ml = np.sum(water_array)
    water_l = water_ml / 1000
    
    return round(water_l, 3)
    
def calculate_power(power_readings):
    """Calculate total energy consumption in Wh and kWh from an array of power readings in W."""    
    time_interval = 1  # Time interval between readings in seconds
    
    # Total energy in kWh
    total_energy_kwh = np.sum(power_readings) * (time_interval / (3600 * 1000))  # Convert to hours and kilowatt-hours
    total_energy_wh = np.sum(power_readings) * (time_interval / 3600)
    
    # print(f"Total Energy Consumption: {total_energy_wh:.3f} Wh")
    # print(f"Total Energy Consumption: {total_energy_kwh:.3f} kWh")
    
    return round(total_energy_wh, 3), round(total_energy_kwh, 3)

def _numeric(series: pd.Series) -> np.ndarray:
    """Return numeric np.array with NaNs for non-numeric."""
    return pd.to_numeric(series, errors="coerce").to_numpy()

def _compute_stats(col: pd.Series) -> dict:
    """Compute all stats in STAT_FUNCS for a column; returns dict of name->value.
       If the column has no valid numbers, returns -1 for all (keeps your convention)."""
    arr = _numeric(col)
    # mask out NaNs
    good = arr[~np.isnan(arr)]
    if good.size == 0:
        return {name: -1 for name in STAT_FUNCS}
    return {name: func(good) for name, func in STAT_FUNCS.items()}

# --- Main loop ---------------------------------------------------------------

for row in aggregated.itertuples(index=True):
    idx = row.Index
    env = getattr(row, "environment", None)
    uid = getattr(row, "unique_monitoring_id", None)
    typ = getattr(row, "type", None)
    path = getattr(row, "file_path", None)


    if env != "laboratory" or pd.isna(uid):
        continue

    # Decide features for this file type
    if typ not in FEATURE_SETS:
        continue
    features = FEATURE_SETS[typ]

    # Build usecols to minimize I/O
    usecols = set(features) | {"power"} | {"water_inlet", "water_outlet"}

    try:
        temp_df = pd.read_csv(path, usecols=list(usecols), low_memory=False)
    except Exception as e:
        # If reading fails, skip this file (or log)
        # print(f"Read failed for {path}: {e}")
        continue

    # --- Energy & water KPIs -------------------------------------------------
    # Energy from power
    if "power" in temp_df.columns:
        power_readings = _numeric(temp_df["power"])
        wh, kwh = calculate_power(power_readings)  # returns (Wh, kWh)
        aggregated.at[idx, "total_energy_consumed_wh"] = wh
        aggregated.at[idx, "total_energy_consumed_kwh"] = kwh

    # Water (wm only)
    if typ == "wm":
        inlet_l = np.nan
        outlet_l = np.nan
        if "water_inlet" in temp_df.columns:
            inlet_l = calculate_water(_numeric(temp_df["water_inlet"]))
            aggregated.at[idx, "total_water_consumed_liters"] = inlet_l
        if "water_outlet" in temp_df.columns:
            outlet_l = calculate_water(_numeric(temp_df["water_outlet"]))
            # If you later want outlet saved:
            # aggregated.at[idx, "total_water_outlet_liters"] = outlet_l

    # --- Statistical features ------------------------------------------------
    for feat in features:
        if feat in temp_df.columns:
            stats = _compute_stats(temp_df[feat])
        else:
            stats = {name: np.nan for name in STAT_FUNCS}  # column missing in file

        for name, value in stats.items():
            aggregated.at[idx, f"{feat}_{name}"] = (
                float(np.round(value, 6)) if isinstance(value, (int, float, np.floating)) else value
            )

    # --- KPIs normalized by load --------------------------------------------
    actual_weight = getattr(row, "actual_weight", 0) or 0
    if typ == "wm":
        # Use already computed wh/inlet_l; fall back to np.nan if missing
        wh_val = aggregated.at[idx, "total_energy_consumed_wh"] if "total_energy_consumed_wh" in aggregated.columns else np.nan
        in_val = aggregated.at[idx, "total_water_consumed_liters"] if "total_water_consumed_liters" in aggregated.columns else np.nan

        if not actual_weight:
            aggregated.at[idx, "e_kpi"] = wh_val
            aggregated.at[idx, "w_kpi"] = in_val
            aggregated.at[idx, "ew_kpi"] = (
                wh_val + 0.78 * in_val if np.isfinite(wh_val) and np.isfinite(in_val) else np.nan
            )
        else:
            e_kpi = wh_val / actual_weight if np.isfinite(wh_val) else np.nan
            w_kpi = in_val / actual_weight if np.isfinite(in_val) else np.nan
            aggregated.at[idx, "e_kpi"] = np.round(e_kpi, 3)
            aggregated.at[idx, "w_kpi"] = np.round(w_kpi, 3)
            aggregated.at[idx, "ew_kpi"] = np.round(
                (e_kpi + 0.78 * w_kpi) if np.isfinite(e_kpi) and np.isfinite(w_kpi) else np.nan, 3
            )
            
        if VERBOSE:
            print(f"Processed WM idx {idx}, UID {uid}, Weight {actual_weight}, Energy {wh_val} Wh, Water {in_val} L")

    elif typ == "dm":
        wh_val = aggregated.at[idx, "total_energy_consumed_wh"] if "total_energy_consumed_wh" in aggregated.columns else np.nan
        if not actual_weight:
            aggregated.at[idx, "e_kpi"] = wh_val
        else:
            e_kpi = wh_val / actual_weight if np.isfinite(wh_val) else np.nan
            aggregated.at[idx, "e_kpi"] = np.round(e_kpi, 3)
        
        if VERBOSE:
            print(f"Processed DM idx {idx}, UID {uid}, Weight {actual_weight}, Energy {wh_val} Wh")

    
if UPDATE:
    aggregated.to_csv("aggregated_data.csv", index=False)
    if VERBOSE:
        print("Aggregated data updated to aggregated_data.csv")